# Интегративный анализ афтершоков: Кластерный метод + Сравнение с другими методами

Этот notebook объединяет основной кластерный анализ из `3_clust_SoCal_Tohoku_2011.ipynb` с сравнительной визуализацией методов из `make_aftershocks_list_Tohoku.ipynb`.

Основной фокус: 
- Кластерный анализ (dNND и dClust) как основной метод.
- Сравнение с методами: Initial catalog, AftIdent, Gardner-Knopoff, Reasenberg Declustering, DBSCAN.

Результат: Расширенная сравнительная карта с 6 панелями (включая кластерный метод).

In [ ]:
# Импорты из обоих notebooks
import pandas as pd
import numpy as np
from math import radians, sin, cos, atan2, sqrt
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import re
from datetime import timedelta
from dateutil.relativedelta import relativedelta
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform

# Дополнительные импорты для кластерного анализа
from matplotlib.colors import LinearSegmentedColormap, to_rgba
import scipy.io
import os

# Импорт внутренних модулей для кластерного анализа (если доступны)
# from src.EqCat import EqCat
# from src import clustering

# Параметры главного толчка (из первого notebook)
earthquake = {
    'event': '2011 Tohoku earthquake',
    'time': '2011 03 11 05:47:32.8',
    'latitude': 37.52,
    'longitude': 143.05,
    'depth_km': 20,
    'Mw': 9.1,
    'scalar_moment_Nm': 5.31e29,
    'strike_deg': 203,
    'dip_deg': 10,
    'rake_deg': 88
}

# Пути к файлам (адаптируйте под вашу структуру)
isc_file_path = 'data/Tohoku_eqs.txt'  # Из первого notebook
mat_file_path = '../data/NEIC_Global_Tohoku_2011.mat'  # Из второго notebook

In [ ]:
# Функция загрузки ISC каталога (из первого notebook)
def load_isc_catalog(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    start_idx = 21
    end_idx = len(lines)
    for i in range(start_idx, len(lines)):
        if lines[i].strip().upper() == 'STOP':
            end_idx = i
            break
    
    events = []
    magnitudes = []
    
    for i in range(start_idx, end_idx):
        line = lines[i].strip()
        if not line or line.startswith('#'):
            continue
            
        if line.endswith(','):
            line = line[:-1]
            
        parts = [part.strip() for part in line.split(',')]
        
        if len(parts) < 9 or not parts[0].replace('.', '').isdigit():
            continue
        
        try:
            event_id = int(parts[0])
            event_type = parts[1].strip()
            author = parts[2].strip()
            date = parts[3].strip()
            time = parts[4].strip()
            
            lat = float(parts[5]) if parts[5] else np.nan
            lon = float(parts[6]) if parts[6] else np.nan
            depth = float(parts[7]) if parts[7] else np.nan
            depfix = parts[8].strip()
            
            events.append({
                'EVENTID': event_id,
                'TYPE': event_type,
                'AUTHOR': author,
                'DATE': date,
                'TIME': time,
                'LAT': lat,
                'LON': lon,
                'DEPTH': depth,
                'DEPFIX': depfix
            })
            
            j = 9
            while j + 2 < len(parts):
                mag_author = parts[j].strip()
                mag_type = parts[j+1].strip()
                mag_value = parts[j+2].strip()
                
                if mag_value and re.match(r'-?\d+(\.\d+)?', mag_value):
                    magnitudes.append({
                        'EVENTID': event_id,
                        'AUTHOR': mag_author,
                        'MAG_TYPE': mag_type,
                        'MAG_VALUE': float(mag_value)
                    })
                
                j += 3
                
        except (ValueError, IndexError) as e:
            print(f"Warning: Error parsing line {i+1}: {e}")
    
    main_df = pd.DataFrame(events)
    magnitudes_df = pd.DataFrame(magnitudes) if magnitudes else pd.DataFrame(columns=['EVENTID', 'AUTHOR', 'MAG_TYPE', 'MAG_VALUE'])
    
    print(f"Successfully loaded {len(main_df)} earthquake events")
    if not magnitudes_df.empty:
        print(f"Loaded {len(magnitudes_df)} magnitude measurements from {magnitudes_df['AUTHOR'].nunique()} agencies")
    
    return main_df, magnitudes_df

# Загрузка ISC данных
main_df, magnitudes_df = load_isc_catalog(isc_file_path)

# Обработка для filtered_df (адаптируйте логику фильтрации из первого notebook)
# Здесь предполагаем, что filtered_df - это отфильтрованный каталог (замените на вашу логику)
filtered_df = main_df.copy()  # Placeholder: добавьте фильтрацию по времени, магнитуде и т.д.
filtered_df['latitude'] = filtered_df['LAT']
filtered_df['longitude'] = filtered_df['LON']
filtered_df['depth'] = filtered_df['DEPTH']
filtered_df['mag'] = magnitudes_df.groupby('EVENTID')['MAG_VALUE'].max().reindex(filtered_df['EVENTID']).values  # Пример объединения магнитуд
filtered_df['time'] = pd.to_datetime(filtered_df['DATE'] + ' ' + filtered_df['TIME'])  # Пример времени

# Placeholder для других DataFrame из первого notebook (aftershocks_ai, etc.)
# Замените на реальные вычисления
aftershocks_ai = filtered_df.copy()  # Placeholder
aftershocks_gk = filtered_df.copy()  # Placeholder
aftershocks_reas = filtered_df.copy()  # Placeholder
aftershocks_dbscan = filtered_df.copy()  # Placeholder

In [ ]:
# Загрузка и обработка для кластерного анализа (из второго notebook)
# Адаптируйте под EqCat, если доступен; иначе используйте загруженные данные
dir_in = '../data'
file_in = 'NEIC_Global_Tohoku_2011.mat'
Mmin, Mmax = 3, None
tmin, tmax = 1900, 2025

# Загрузка MAT файла (scipy.io)
mat_data = scipy.io.loadmat(f"{dir_in}/{file_in}")
# Предполагаем структуру: mat_data['eqCat'] или подобное; адаптируйте
# eqCat = EqCat()
# eqCat.loadMatBin(f"{dir_in}/{file_in}")
# eqCat.selectEvents(Mmin, Mmax, 'Mag')
# eqCat.selectEvents(tmin, tmax, 'Time')

# Placeholder для dNND и dClust (замените на реальные вычисления из clustering)
# dNND = ...  # Результат расчета расстояний ближайшего соседа
# dClust = ...  # Словарь кластеров
# dPar = {'eta_0': ...}  # Параметры

# Создание DataFrame для кластерного метода
aftershocks_clust = filtered_df.copy()  # Placeholder: отфильтруйте по кластерам
# Логика: aftershocks_clust = pd.DataFrame() для всех событий в кластерах
# Например:
# for famID, indices in dClust.items():
#     sel = np.isin(eqCat.data['N'], indices)
#     aftershocks_clust = pd.concat([aftershocks_clust, filtered_df[sel]])

print(f"Clustered events: {len(aftershocks_clust)}")

In [ ]:
# Расширенная функция plotting (адаптирована из первого notebook + кластерный метод)
def plot_integrated_aftershock_methods_map(
    full_df, ai_df, gk_df, reas_df, dbscan_df, clust_df, mainshock_lat, mainshock_lon
):
    fig, axs = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': ccrs.PlateCarree()})
    methods = [
        ("Initial catalog", full_df, axs[0, 0]),
        ("AftIdent", ai_df, axs[0, 1]),
        ("Gardner-Knopoff", gk_df, axs[0, 2]),
        ("Reasenberg Declustering", reas_df, axs[1, 0]),
        ("DBSCAN Clustering", dbscan_df, axs[1, 1]),
        ("NND-based Clustering (Main)", clust_df, axs[1, 2])  # Новый: основной кластерный метод
    ]

    # Границы карты (из первого notebook)
    boundaries_df = pd.DataFrame({  # Placeholder: добавьте реальные границы
        'longitude': [138, 147, 147, 138, 138],
        'latitude': [34, 34, 42, 42, 34]
    })

    for title, df, ax in methods:
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.gridlines(draw_labels=True)

        # Plot aftershocks (стандартный scatter)
        if 'mag' in df.columns:  # Если есть магнитуда для размера
            sizes = [mag_to_size(m) for m in df['mag']]
            ax.scatter(df['longitude'], df['latitude'], color='orange', s=sizes, label='Aftershocks', alpha=0.7)
        else:
            ax.scatter(df['longitude'], df['latitude'], color='orange', s=20, label='Aftershocks', alpha=0.7)

        # Plot mainshock
        ax.scatter(mainshock_lon, mainshock_lat, color='red', marker='*', s=100, label='Mainshock', zorder=5)

        # Boundary line
        ax.plot(boundaries_df['longitude'], boundaries_df['latitude'], color='black', label='Boundary Line')

        # Extent
        min_lon, max_lon = 138, 147
        min_lat, max_lat = 34, 42
        ax.set_extent([min_lon, max_lon, min_lat, max_lat], crs=ccrs.PlateCarree())

        ax.set_title(title)
        ax.legend(loc='lower left')

    # Для кластерного метода: добавьте цвет по глубине (из второго notebook)
    # В панели "NND-based Clustering (Main)" добавьте cmap по глубине
    # (адаптируйте в цикле выше, если нужно)

    plt.suptitle("Integrated Comparison: Clustering Methods for Tohoku Aftershocks", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# Функция mag_to_size (из второго notebook)
def mag_to_size(mag):
    if mag < 4:
        return 10
    elif 4 <= mag < 5:
        return 40
    elif 5 <= mag < 6:
        return 100
    elif 6 <= mag < 7:
        return 200
    elif 7 <= mag < 8:
        return 350
    else:
        return 600

# Вызов функции
plot_integrated_aftershock_methods_map(
    full_df=filtered_df,
    ai_df=aftershocks_ai,
    gk_df=aftershocks_gk,
    reas_df=aftershocks_reas,
    dbscan_df=aftershocks_dbscan,
    clust_df=aftershocks_clust,
    mainshock_lat=earthquake['latitude'],
    mainshock_lon=earthquake['longitude']
)

In [ ]:
# Сохранение фигуры
plt.savefig('integrated_aftershocks_comparison_Tohoku.png', dpi=300, bbox_inches='tight')
print("Integrated map saved as 'integrated_aftershocks_comparison_Tohoku.png'")